In [23]:
library(hictoolsr)
library(dbscan)
library(tidyverse)
library(GenomicRanges)
library(InteractionSet)
library(readr)


Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:lubridate’:

    intersect, setdiff, union


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, saveRDS, setdiff,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:lubridate’:

    second, second<-




In [20]:
loops <- read_csv("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/jupyter_notes/merged_loops.csv")

Rows: 32277 Columns: 16
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (6): chr1, chr2, status, condition, comp_switch_A1, comp_switch_A2
dbl (10): start1, end1, start2, end2, ctrl_signal, rbp1_signal, log2FC, log2...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [21]:
head(loops)

chr1,start1,end1,chr2,start2,end2,status,ctrl_signal,rbp1_signal,log2FC,log2FC_clipped,condition,distance,loop_id,comp_switch_A1,comp_switch_A2
<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>
chr1,1952500,1957500,chr1,2042500,2047500,shared,NA,NA,0.0000000,0.0000000,shared,90000,0,NA,NA
chr1,2202500,2207500,chr1,2382500,2387500,shared,NA,NA,0.0000000,0.0000000,shared,180000,1,NA,NA
chr1,2412500,2417500,chr1,2552500,2557500,shared,NA,NA,0.0000000,0.0000000,shared,140000,2,NA,NA
chr1,3490000,3495000,chr1,3615000,3620000,shared,NA,NA,0.0000000,0.0000000,shared,125000,3,NA,NA
chr1,3565000,3570000,chr1,3615000,3620000,shared,NA,NA,0.0000000,0.0000000,shared,50000,4,NA,NA
chr1,3912500,3917500,chr1,4822500,4827500,shared,0.006495129,0.007784711,0.2612852,0.2612852,shared,910000,5,stable,stable


In [24]:
anchor1 <- GRanges(
  seqnames = loops$chr1,
  ranges = IRanges(start = loops$start1, end = loops$end1)
)

anchor2 <- GRanges(
  seqnames = loops$chr2,
  ranges = IRanges(start = loops$start2, end = loops$end2)
)

In [25]:
# create ginteractions obj
gi <- GInteractions(anchor1, anchor2)

In [26]:
gi

GInteractions object with 32277 interactions and 0 metadata columns:
          seqnames1           ranges1     seqnames2           ranges2
              <Rle>         <IRanges>         <Rle>         <IRanges>
      [1]      chr1   1952500-1957500 ---      chr1   2042500-2047500
      [2]      chr1   2202500-2207500 ---      chr1   2382500-2387500
      [3]      chr1   2412500-2417500 ---      chr1   2552500-2557500
      [4]      chr1   3490000-3495000 ---      chr1   3615000-3620000
      [5]      chr1   3565000-3570000 ---      chr1   3615000-3620000
      ...       ...               ... ...       ...               ...
  [32273]      chrY 10945000-10950000 ---      chrY 11290000-11295000
  [32274]      chrY 10982500-10987500 ---      chrY 11292500-11297500
  [32275]      chrY 11292500-11297500 ---      chrY 11722500-11727500
  [32276]      chrY 11530000-11535000 ---      chrY 11760000-11765000
  [32277]      chrY 19432500-19437500 ---      chrY 20742500-20747500
  -------
  regions: 

In [28]:
## Hi-C file paths

hicFiles <- c( "/usr/users/papantonis1/aman/microc_data/nadine_macro/ctrl.hic", "/usr/users/papantonis1/aman/microc_data/nadine_macro/RBP1.hic")

## Extract Hi-C counts between loop pixels
loopCounts <- extractCounts(bedpe = gi,
              hic = hicFiles,
              chroms = c(1:22, "X"),
              res = 10e3,
              norm = 'NONE',
              matrix = 'observed')

Warning message:
“bedpe is not binned correctly.  Binning each anchor at center position.For more options use the binBedpe() function.”


In [29]:
# check the column names
colnames(mcols(loopCounts))

[1] "ctrl.hic" "RBP1.hic"

In [30]:
head(loopCounts)

GInteractions object with 6 interactions and 2 metadata columns:
      seqnames1         ranges1     seqnames2         ranges2 |  ctrl.hic
          <Rle>       <IRanges>         <Rle>       <IRanges> | <numeric>
  [1]      chr1 1950000-1960000 ---      chr1 2040000-2050000 |        34
  [2]      chr1 2200000-2210000 ---      chr1 2380000-2390000 |        21
  [3]      chr1 2410000-2420000 ---      chr1 2550000-2560000 |        18
  [4]      chr1 3490000-3500000 ---      chr1 3610000-3620000 |        58
  [5]      chr1 3560000-3570000 ---      chr1 3610000-3620000 |        72
  [6]      chr1 3910000-3920000 ---      chr1 4820000-4830000 |        11
       RBP1.hic
      <numeric>
  [1]        25
  [2]        22
  [3]        14
  [4]        40
  [5]        45
  [6]        11
  -------
  regions: 35358 ranges and 0 metadata columns
  seqinfo: 24 sequences from an unspecified genome; no seqlengths

In [32]:
## Load deseq
library(DESeq2)

## Isolate count matrix
cnts <- 
  mcols(loopCounts)[grep("ctrl|RBP1", colnames(mcols(loopCounts)))] |>
  as.matrix()

head(cnts)

ctrl.hic,RBP1.hic
34,25
21,22
18,14
58,40
72,45
11,11


In [34]:
colData <- data.frame(
  condition = c("CTRL", "RBP1")
)
colData

condition
<chr>
CTRL
RBP1


In [36]:
rownames(colData) <- colnames(cnts)

In [38]:
## Build DESeq data set
dds <- 
  DESeqDataSetFromMatrix(countData = cnts,
                         colData = colData,
                         design = ~ condition)

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”


In [39]:
res <-
  DESeq(dds) |>
  lfcShrink(coef = "ctrl vs eed", type="apeglm")

estimating size factors

estimating dispersions



ERROR: Error in checkForExperimentalReplicates(object, modelMatrix): 

  The design matrix has the same number of samples and coefficients to fit,
  so estimation of dispersion is not possible. Treating samples
  as replicates was deprecated in v1.20 and no longer supported since v1.22.




### Only possible with replicates

In [40]:
sessionInfo()

R version 4.4.3 (2025-02-28)
Platform: x86_64-conda-linux-gnu
Running under: Rocky Linux 8.10 (Green Obsidian)

Matrix products: default
BLAS/LAPACK: /home/uni08/papantonis1/anaconda3/envs/hictoolsr_aman/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] DESeq2_1.46.0               InteractionSet_1.34.0      
 [3] SummarizedExperiment_1.36.0 Biobase_2.66.0             
 [5] MatrixGenerics_1.18.0       matrixStats_1.5.0          
 [7] GenomicRanges_1.58.0        GenomeInfoDb_1.42.0        
